# Final Training: NewsBERT-germ-210m
## EuroBERT-210M mit optimierten HPT-Parametern (automatisch aus Optuna DB)

Trainiert das finale EuroBERT-210M Modell mit den besten Hyperparametern
aus der Optuna Phase-1 Studie und pusht es als `Zorryy/NewsBERT-germ-210m`
auf HuggingFace mit vollstaendigem Model Card.

**Features:**
- Hyperparameter werden automatisch aus der Optuna SQLite-DB geladen
- NaN-Detection mit automatischem Retry (bis zu 3 Versuche, verschiedene Seeds)
- Early Stopping (patience=3), max 15 Epochs
- Evaluation auf allen Klassen mit Per-Class Metrics
- Model Card README mit Performance, HPT-Params, Usage Example

**Voraussetzung:** GPU-Runtime (L4 empfohlen), `HF_TOKEN` in Colab Secrets.

In [ ]:
# === SETUP ===
import os, sys

REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

!pip install -q transformers[sentencepiece] datasets huggingface_hub \
    scikit-learn matplotlib seaborn tqdm pandas accelerate evaluate \
    optuna

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PIPELINE_DIR = f"{REPO}/Python/classification_pipeline"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

HPT_DIR = f"{REPO}/Python/classification_pipeline/hyper_parameter_tuning"
if HPT_DIR not in sys.path:
    sys.path.insert(0, HPT_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)
import hpt_utils as hu
importlib.reload(hu)

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

print(f"Reports-Ordner: {pu.REPORTS_DIR}")
print("Setup abgeschlossen.")

In [ ]:
# === BEST HYPERPARAMETERS AUS OPTUNA DB ===
import optuna
from pathlib import Path

# DB-Pfad (Google Drive, identisch mit HPT-Notebooks)
# Falls die DB einen anderen Namen hat (z.B. _002), hier anpassen:
OPTUNA_DB_NAME = "eurobert_210m_hpt_phase1_001.db"
OPTUNA_DB_PATH = Path(pu.REPORTS_DIR) / OPTUNA_DB_NAME

if not OPTUNA_DB_PATH.exists():
    raise FileNotFoundError(
        f"Optuna DB nicht gefunden: {OPTUNA_DB_PATH}\n"
        f"Bitte HPT Phase 1 zuerst ausfuehren oder OPTUNA_DB_NAME anpassen."
    )

storage_url = f"sqlite:///{OPTUNA_DB_PATH}"
study = optuna.load_study(
    study_name="eurobert_210m_hpt_phase1",
    storage=storage_url,
)

print(f"Optuna DB geladen: {OPTUNA_DB_PATH}")
print(f"Beste Trial: {study.best_trial.number}")
print(f"Bester F1 Macro (CV Mean): {study.best_value:.4f}")

best_params = study.best_params
print(f"\nBeste Hyperparameter:")
for key, val in best_params.items():
    print(f"  {key}: {val}")

# Extrahierte Parameter
LEARNING_RATE = best_params["learning_rate"]
LR_SCHEDULER_TYPE = best_params["lr_scheduler_type"]
NUM_EPOCHS_HPT = best_params["num_train_epochs"]
BATCH_SIZE_TRAIN = best_params["per_device_train_batch_size"]
WARMUP_RATIO = best_params["warmup_ratio"]
WEIGHT_DECAY = best_params["weight_decay"]
LABEL_SMOOTHING = best_params["label_smoothing_factor"]

In [ ]:
# === MODEL & TRAINING CONFIG ===
import torch
import numpy as np

MODEL_ID = "EuroBERT/EuroBERT-210m"
MODEL_SHORT_NAME = "NewsBERT_germ_210m"
MODEL_TYPE = "fine-tuned"
REPO_NAME = "Zorryy/NewsBERT-germ-210m"

# ----- Feste Parameter -----
MAX_EPOCHS = 15
NUM_EPOCHS = min(NUM_EPOCHS_HPT, MAX_EPOCHS)
MAX_LENGTH = 2048
EFFECTIVE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = max(1, EFFECTIVE_BATCH_SIZE // BATCH_SIZE_TRAIN)
RANDOM_SEED = 42
TEST_PER_CLASS = 30
VAL_FRACTION = 0.2
EARLY_STOPPING_PATIENCE = 3
LOGGING_STEPS = 10
MAX_GRAD_NORM = 0.5
MAX_NAN_RETRIES = 3

# ----- GPU-adaptive Einstellungen -----
if not torch.cuda.is_available():
    raise RuntimeError("GPU benoetigt! Bitte Colab Runtime aendern.")

_gpu_cap = torch.cuda.get_device_capability()
_gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

if _gpu_cap[0] >= 8:  # Ampere+ (L4, A100)
    USE_BF16 = True
    USE_FP16 = False
else:  # T4
    USE_BF16 = False
    USE_FP16 = True

if _gpu_mem >= 40:
    BATCH_SIZE_EVAL = 32
elif _gpu_mem >= 20:
    BATCH_SIZE_EVAL = 16
else:
    BATCH_SIZE_EVAL = 8

OPTIM = "adamw_torch_fused"

ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

MODEL_INFO = {
    "huggingface_id": MODEL_ID,
    "language": "Multilingual (inkl. Deutsch)",
    "max_tokens": MAX_LENGTH,
    "parameters": "210M",
    "notes": (
        "EuroBERT-210m fine-tuned mit Optuna-optimierten Hyperparametern. "
        "Finale Version fuer Bundestagswahl-2025-Nachrichtenklassifikation."
    ),
}

print(f"GPU: {torch.cuda.get_device_name(0)} ({_gpu_mem:.1f} GB, CC {_gpu_cap[0]}.{_gpu_cap[1]})")
print(f"  BF16={USE_BF16}, FP16={USE_FP16}, Eval Batch={BATCH_SIZE_EVAL}")
print(f"\nHPT-Parameter (aus Optuna DB):")
print(f"  LR:              {LEARNING_RATE:.2e}")
print(f"  Scheduler:       {LR_SCHEDULER_TYPE}")
print(f"  Epochs:          {NUM_EPOCHS} (HPT={NUM_EPOCHS_HPT}, max={MAX_EPOCHS})")
print(f"  Batch (train):   {BATCH_SIZE_TRAIN}")
print(f"  Grad Accum:      {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effektive BS:    {EFFECTIVE_BATCH_SIZE}")
print(f"  Warmup Ratio:    {WARMUP_RATIO:.4f}")
print(f"  Weight Decay:    {WEIGHT_DECAY:.4f}")
print(f"  Label Smoothing: {LABEL_SMOOTHING:.4f}")
print(f"  max_grad_norm:   {MAX_GRAD_NORM}")
print(f"  Early Stopping:  patience={EARLY_STOPPING_PATIENCE}")
print(f"  NaN Retries:     {MAX_NAN_RETRIES}")

In [ ]:
# === DATEN LADEN & CUSTOM SPLIT ===
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

np.random.seed(RANDOM_SEED)

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()
all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)

print(f"Gesamtpool gelabelter Artikel: {len(all_labelled)}")
print(f"Klassen: {all_labelled['label'].nunique()}\n")

# Test-Split (fix, stratifiziert) — identisch mit HPT Notebooks
test_indices = []
rest_indices = []

for label in ALL_LABELS:
    label_mask = all_labelled["label"] == label
    label_indices = all_labelled[label_mask].index.tolist()
    n_total = len(label_indices)
    if n_total < 60:
        n_test = n_total // 2
        print(f"  {label}: nur {n_total} Artikel -> {n_test} fuer Test")
    else:
        n_test = TEST_PER_CLASS
    np.random.shuffle(label_indices)
    test_indices.extend(label_indices[:n_test])
    rest_indices.extend(label_indices[n_test:])

test_df = all_labelled.loc[test_indices].reset_index(drop=True)
rest_df = all_labelled.loc[rest_indices].reset_index(drop=True)

# Train/Val Split (stratifiziert, 80/20)
class_counts = rest_df["label"].value_counts()
small_classes = class_counts[class_counts < 2].index.tolist()

if small_classes:
    small_mask = rest_df["label"].isin(small_classes)
    train_small = rest_df[small_mask]
    rest_for_split = rest_df[~small_mask]
else:
    train_small = pd.DataFrame(columns=rest_df.columns)
    rest_for_split = rest_df

train_main, val_df = train_test_split(
    rest_for_split, test_size=VAL_FRACTION,
    stratify=rest_for_split["label"], random_state=RANDOM_SEED,
)
train_df = pd.concat([train_main, train_small], ignore_index=True)
val_df = val_df.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"  Train:      {len(train_df):>5}")
print(f"  Validation: {len(val_df):>5}")
print(f"  Test:       {len(test_df):>5}")
print(f"  Gesamt:     {len(train_df) + len(val_df) + len(test_df):>5}")
print(f"{'='*50}")

split_config = {
    "dataset_id": pu.DATASET_ID,
    "split_mode": "custom_finetune",
    "test_per_class": TEST_PER_CLASS,
    "val_fraction": VAL_FRACTION,
    "random_seed": RANDOM_SEED,
    "train_size": len(train_df),
    "eval_size": len(val_df),
    "test_size": len(test_df),
    "raw_size": 0,
}

In [ ]:
# === LABEL ENCODING ===
label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

for name, _df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    assert _df["label_id"].isna().sum() == 0, f"Unbekannte Labels in {name}!"

print("Label-Mapping:")
for label, idx in label2id.items():
    print(f"  {idx:>2}: {label}")
print(f"\nAnzahl Klassen: {len(ALL_LABELS)}")

In [ ]:
# === TOKENISIERUNG ===
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def tokenize_fn(examples):
    return tokenizer(examples["text"], max_length=MAX_LENGTH, truncation=True)

train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
val_dataset = Dataset.from_pandas(val_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))

print("Tokenisiere...")
train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# === ROPE FIX & MODEL FACTORY ===
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoConfig
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

# EuroBERT RoPE Fix (identisch mit allen EuroBERT-Notebooks)
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init


def create_model(seed=42):
    """Erstellt ein frisches Modell mit deterministischer Initialisierung.

    NaN-Prevention:
    1. Deterministisches Seeding -> reproduzierbare Classifier-Head-Init
    2. Re-Init mit kleinerem std (0.002) -> verhindert Logit-Overflow in BF16
    """
    torch.manual_seed(seed)

    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config.num_labels = len(ALL_LABELS)
    config.id2label = id2label
    config.label2id = label2id

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, config=config,
        ignore_mismatched_sizes=True, trust_remote_code=True,
    )

    # Classifier Head stabilisieren: kleinere Init verhindert Logit-Overflow in BF16
    for name, module in model.named_modules():
        if name in ("dense", "classifier") and isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.002)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    return model


print("ROPE Fix angewendet.")
print("create_model() definiert.")

In [ ]:
# === COMPUTE METRICS ===
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    classification_report,
)

def compute_metrics(eval_pred):
    """Metrics fuer Trainer: Aggregate + Per-Class (fuer HPTMetricsCallback)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Store fuer HPTMetricsCallback (Confusion Matrix)
    hu._prediction_store["current"] = {"preds": preds, "labels": labels}

    result = {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

    # Per-class F1/Precision/Recall (fuer HPTMetricsCallback Epoch-Tracking)
    report = classification_report(
        labels, preds,
        labels=list(range(len(ALL_LABELS))),
        target_names=ALL_LABELS,
        output_dict=True, zero_division=0,
    )
    for label_name in ALL_LABELS:
        safe = hu._safe_label_key(label_name)
        if label_name in report:
            result[f"class_{safe}_f1"] = report[label_name]["f1-score"]
            result[f"class_{safe}_precision"] = report[label_name]["precision"]
            result[f"class_{safe}_recall"] = report[label_name]["recall"]

    return result

print("compute_metrics definiert (mit Per-Class Tracking).")

In [ ]:
# === TRAINING MIT NaN AUTO-RETRY ===
import gc
import math
import shutil
from transformers import (
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding,
)

OUTPUT_DIR = "/content/final_model_output"


def run_training(attempt_seed):
    """Einzelner Trainings-Versuch. Gibt (trainer, success) zurueck."""

    print(f"\n{'='*60}")
    print(f"  Training-Versuch mit Seed {attempt_seed}")
    print(f"{'='*60}")

    # VRAM aufraeumen
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Frisches Modell
    model = create_model(seed=attempt_seed)
    model = model.to(torch.device("cuda"))

    # Warmup Steps berechnen
    steps_per_epoch = math.ceil(
        len(train_dataset) / (BATCH_SIZE_TRAIN * GRADIENT_ACCUMULATION_STEPS)
    )
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    # NaN Detection Callback
    nan_callback = hu.HPTMetricsCallback(
        trial_number=0,
        fold_idx=attempt_seed,
        all_labels=ALL_LABELS,
        id2label=id2label,
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        per_device_train_batch_size=BATCH_SIZE_TRAIN,
        per_device_eval_batch_size=BATCH_SIZE_EVAL,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        label_smoothing_factor=LABEL_SMOOTHING,
        fp16=USE_FP16,
        bf16=USE_BF16,
        gradient_checkpointing=False,
        optim=OPTIM,
        max_grad_norm=MAX_GRAD_NORM,
        group_by_length=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=3,
        report_to="none",
        seed=attempt_seed,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
            nan_callback,
        ],
    )

    print(f"  Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE:.2e}, Warmup: {warmup_steps} steps")
    print(f"  Batch: {BATCH_SIZE_TRAIN} x {GRADIENT_ACCUMULATION_STEPS} = {EFFECTIVE_BATCH_SIZE}")

    trainer.train()

    # NaN / Collapse Check
    if nan_callback.nan_detected or nan_callback.lr_zero_detected:
        reason = "NaN" if nan_callback.nan_detected else "LR=0"
        print(f"  [{reason} DETECTED] Training abgebrochen.")
        return trainer, False

    # Final eval sanity check
    eval_result = trainer.evaluate()
    eval_f1 = eval_result.get("eval_f1_macro", 0)
    eval_loss = eval_result.get("eval_loss", 0)

    if (eval_loss is not None and (math.isnan(eval_loss) or math.isinf(eval_loss))) \
       or eval_f1 < 0.05:
        print(f"  [COLLAPSE] Eval F1={eval_f1:.4f}, Loss={eval_loss} -> Training gescheitert.")
        return trainer, False

    print(f"  Training erfolgreich! Eval F1 Macro: {eval_f1:.4f}")
    return trainer, True


# === NaN RETRY LOOP ===
timer = pu.ExperimentTimer()
trainer = None
training_success = False

with timer:
    for attempt in range(MAX_NAN_RETRIES):
        attempt_seed = RANDOM_SEED + attempt * 1000
        print(f"\n>>> Versuch {attempt + 1}/{MAX_NAN_RETRIES} (seed={attempt_seed})")

        # Cleanup previous attempt
        if trainer is not None:
            del trainer
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        if os.path.exists(OUTPUT_DIR):
            shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

        trainer, training_success = run_training(attempt_seed)

        if training_success:
            print(f"\n>>> Training erfolgreich nach {attempt + 1} Versuch(en)!")
            break
        else:
            print(f"\n>>> Versuch {attempt + 1} gescheitert. ", end="")
            if attempt < MAX_NAN_RETRIES - 1:
                print("Starte naechsten Versuch mit neuem Seed...")
            else:
                print("Alle Versuche gescheitert!")

if not training_success:
    raise RuntimeError(
        f"Training nach {MAX_NAN_RETRIES} Versuchen gescheitert! "
        f"Moegliche Ursachen: NaN, LR=0, Model Collapse."
    )

print(f"\nTraining abgeschlossen: {timer.duration_formatted}")
print(f"Bestes Modell: {trainer.state.best_model_checkpoint}")
print(f"Beste Metrik (F1 Macro): {trainer.state.best_metric:.4f}")

In [ ]:
# === TRAINING-VERLAUF ===
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]

eval_logs = [e for e in log_history if "eval_loss" in e]
eval_epochs = [e["epoch"] for e in eval_logs]
eval_losses = [e["eval_loss"] for e in eval_logs]
eval_f1 = [e.get("eval_f1_macro", 0) for e in eval_logs]
eval_acc = [e.get("eval_accuracy", 0) for e in eval_logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.plot(train_steps, train_losses, alpha=0.4, label="Train Loss", color="steelblue")
eval_steps = [e.get("step", 0) for e in eval_logs]
ax1.plot(eval_steps, eval_losses, "o-", label="Eval Loss", color="orangered", linewidth=2)
ax1.set_xlabel("Steps")
ax1.set_ylabel("Loss")
ax1.set_title("Training vs. Eval Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(eval_epochs, eval_f1, "o-", label="F1 Macro", linewidth=2)
ax2.plot(eval_epochs, eval_acc, "s-", label="Accuracy", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Eval-Metriken pro Epoch")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle("NewsBERT-germ-210m Final Training", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nEval-Metriken pro Epoch:")
print(f"{'Epoch':>6} {'Loss':>8} {'F1 Macro':>10} {'Accuracy':>10}")
print("-" * 40)
for log in eval_logs:
    print(f"{log['epoch']:>6.0f} {log['eval_loss']:>8.4f} "
          f"{log.get('eval_f1_macro', 0):>10.4f} {log.get('eval_accuracy', 0):>10.4f}")

In [ ]:
# === EVALUATION AUF TEST-SET ===
print("Evaluation auf Test-Set mit bestem Modell...")

test_preds = trainer.predict(test_dataset)
pred_ids = np.argmax(test_preds.predictions, axis=-1)
pred_labels = [id2label[i] for i in pred_ids]
true_labels = [id2label[i] for i in test_preds.label_ids]

metrics = pu.evaluate(
    true_labels, pred_labels,
    labels=ALL_LABELS, experiment_name="test",
)
pu.print_metrics(metrics, "NewsBERT-germ-210m -- Test Split")

In [ ]:
# === CONFUSION MATRIX ===
pu.plot_confusion_matrix(metrics, title="NewsBERT-germ-210m (Test)")

In [ ]:
# === PER-CLASS METRICS ===
pc_df = metrics["per_class_df"].copy().sort_values("F1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(pc_df))
bar_h = 0.25

ax.barh(y_pos - bar_h, pc_df["Precision"], bar_h, label="Precision", color="#2196F3", alpha=0.85)
ax.barh(y_pos, pc_df["Recall"], bar_h, label="Recall", color="#FF9800", alpha=0.85)
ax.barh(y_pos + bar_h, pc_df["F1"], bar_h, label="F1", color="#4CAF50", alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(pc_df["Label"])
ax.set_xlabel("Score")
ax.set_title("Per-Class Metrics: NewsBERT-germ-210m (Final)", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)

for i, (_, row) in enumerate(pc_df.iterrows()):
    ax.text(row["F1"] + 0.01, y_pos[i] + bar_h, f"{row['F1']:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# === BEST EPOCH: PER-CLASS PERFORMANCE ===
print(f"Best Checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best Epoch Metric (val F1 Macro): {trainer.state.best_metric:.4f}\n")

# Test-Set Ergebnisse (bestes Modell via load_best_model_at_end)
pc = metrics["per_class_df"]

print(f"{'Label':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>8}")
print("=" * 70)
for _, row in pc.iterrows():
    print(f"{row['Label']:<30} {row['Precision']:>10.4f} {row['Recall']:>10.4f} "
          f"{row['F1']:>10.4f} {int(row['Support']):>8}")
print("=" * 70)
print(f"{'F1 Macro':<30} {'':>10} {'':>10} {metrics['f1_macro']:>10.4f}")
print(f"{'F1 Weighted':<30} {'':>10} {'':>10} {metrics['f1_weighted']:>10.4f}")
print(f"{'Accuracy':<30} {'':>10} {'':>10} {metrics['accuracy']:>10.4f}")

# Validation-Verlauf: beste Epoch identifizieren
best_epoch_idx = np.argmax([e.get("eval_f1_macro", 0) for e in eval_logs])
best_epoch_log = eval_logs[best_epoch_idx]
print(f"\nBeste Validation-Epoch: {best_epoch_log['epoch']:.0f}")
print(f"  Val Loss:     {best_epoch_log['eval_loss']:.4f}")
print(f"  Val F1 Macro: {best_epoch_log.get('eval_f1_macro', 0):.4f}")
print(f"  Val Accuracy: {best_epoch_log.get('eval_accuracy', 0):.4f}")

In [ ]:
# === REPORT ===
training_params = {
    "source": "Optuna HPT Phase 1",
    "optuna_db": str(OPTUNA_DB_PATH.name),
    "optuna_best_trial": study.best_trial.number,
    "optuna_best_cv_f1": round(study.best_value, 4),
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_eval": BATCH_SIZE_EVAL,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing_factor": LABEL_SMOOTHING,
    "max_length": MAX_LENGTH,
    "max_grad_norm": MAX_GRAD_NORM,
    "bf16": USE_BF16,
    "fp16": USE_FP16,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "gradient_checkpointing": False,
    "optimizer": OPTIM,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": round(trainer.state.best_metric, 4) if trainer.state.best_metric else None,
}

config_dict = trainer.model.config.to_dict()
model_config = {}
for field in ["architectures", "model_type", "hidden_size", "num_hidden_layers",
              "num_attention_heads", "vocab_size", "max_position_embeddings"]:
    if field in config_dict:
        val = config_dict[field]
        if field == "architectures" and isinstance(val, list):
            val = val[0] if len(val) == 1 else ", ".join(val)
        model_config[field] = val

report_path = pu.generate_report(
    model_name=MODEL_SHORT_NAME,
    model_type=MODEL_TYPE,
    metrics=metrics,
    timer=timer,
    model_info=MODEL_INFO,
    candidate_labels=ALL_LABELS,
    split_config=split_config,
    label_mapping={l: l for l in ALL_LABELS},
    model_config=model_config,
    training_params=training_params,
    experiment_notes=(
        f"FINAL MODEL: NewsBERT-germ-210m. EuroBERT-210M fine-tuned mit "
        f"Optuna-optimierten Hyperparametern (Trial {study.best_trial.number}, "
        f"CV F1={study.best_value:.4f}). "
        f"Training auf handannotierten Nachrichtenartikeln zur Bundestagswahl 2025. "
        f"Custom Split: {TEST_PER_CLASS} Test/Klasse, Rest "
        f"{int((1-VAL_FRACTION)*100)}/{int(VAL_FRACTION*100)} Train/Val."
    ),
)
print(f"Report gespeichert: {report_path}")

In [ ]:
# === MODEL CARD README ===
from huggingface_hub import HfApi

pc = metrics["per_class_df"]

# Per-Class Tabelle fuer Model Card
per_class_rows = ""
for _, row in pc.iterrows():
    per_class_rows += (
        f"| {row['Label']} | {row['Precision']:.4f} "
        f"| {row['Recall']:.4f} | {row['F1']:.4f} | {int(row['Support'])} |\n"
    )

# Kategorien-Liste
categories_list = "\n".join(f"{i+1}. {label}" for i, label in enumerate(ALL_LABELS))

model_card = f"""---
language:
- de
- multilingual
license: apache-2.0
tags:
- text-classification
- news-classification
- german
- eurobert
- fine-tuned
datasets:
- Zorryy/news_articles_2025_elections_germany
base_model: EuroBERT/EuroBERT-210m
metrics:
- f1
- accuracy
pipeline_tag: text-classification
---

# NewsBERT-germ-210m

German news article classifier based on [EuroBERT-210m](https://huggingface.co/EuroBERT/EuroBERT-210m),
fine-tuned on hand-annotated news articles about the 2025 German federal election (Bundestagswahl).

## Model Description

- **Base Model:** EuroBERT/EuroBERT-210m (210M parameters)
- **Task:** Multi-class text classification (13 categories)
- **Language:** German
- **Training Data:** Hand-annotated German news articles ({split_config['train_size']} train, {split_config['eval_size']} validation, {split_config['test_size']} test)
- **Dataset:** [Zorryy/news_articles_2025_elections_germany](https://huggingface.co/datasets/Zorryy/news_articles_2025_elections_germany)

## Performance (Test Set)

| Metric | Score |
|---|---|
| **F1 Macro** | **{metrics['f1_macro']:.4f}** |
| F1 Weighted | {metrics['f1_weighted']:.4f} |
| Precision Macro | {metrics['precision_macro']:.4f} |
| Recall Macro | {metrics['recall_macro']:.4f} |
| Accuracy | {metrics['accuracy']:.4f} |

### Per-Class Performance

| Category | Precision | Recall | F1 | Support |
|---|---|---|---|---|
{per_class_rows}
## Hyperparameters

Optimized via Optuna TPE sampler with 3-fold stratified cross-validation
(Best Trial: {study.best_trial.number}, CV F1 Macro: {study.best_value:.4f}).

| Parameter | Value |
|---|---|
| Learning Rate | {LEARNING_RATE:.2e} |
| LR Scheduler | {LR_SCHEDULER_TYPE} |
| Epochs | {NUM_EPOCHS} |
| Batch Size (per device) | {BATCH_SIZE_TRAIN} |
| Effective Batch Size | {EFFECTIVE_BATCH_SIZE} |
| Warmup Ratio | {WARMUP_RATIO:.4f} |
| Weight Decay | {WEIGHT_DECAY:.4f} |
| Label Smoothing | {LABEL_SMOOTHING:.4f} |
| Max Sequence Length | {MAX_LENGTH} |
| Gradient Clipping | {MAX_GRAD_NORM} |
| Optimizer | {OPTIM} |
| Early Stopping | patience={EARLY_STOPPING_PATIENCE} |
| Mixed Precision | BF16={USE_BF16}, FP16={USE_FP16} |

## Training Data

The model was trained on **hand-annotated** German news articles related to the
2025 German federal election. Articles were manually labeled into 13 topic categories
by human annotators. The dataset is available at
[Zorryy/news_articles_2025_elections_germany](https://huggingface.co/datasets/Zorryy/news_articles_2025_elections_germany).

**Important:** All labels are hand-annotated (not machine-generated), ensuring high label quality.

## Usage

```python
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="{REPO_NAME}",
    trust_remote_code=True,
)

text = "Die Bundesregierung plant neue Massnahmen zur Reduzierung der CO2-Emissionen."
result = classifier(text)
print(result)
# [{{"label": "Klima / Energie", "score": 0.95}}]
```

## Categories

The model classifies articles into 13 categories:

{categories_list}

## Limitations

- Trained specifically on German news articles about the 2025 Bundestagswahl
- May not generalize well to other domains, time periods, or languages
- Performance varies by category (see per-class metrics above)

## Citation

If you use this model, please cite the underlying dataset and base model.
"""

print("Model Card erstellt.")
print(f"Laenge: {len(model_card)} Zeichen")

In [ ]:
# === MODELL AUF HUGGINGFACE PUSHEN ===
api = HfApi()

# 1. Push model + tokenizer + training_config.json
url = pu.upload_model_to_hub(
    model=trainer.model,
    tokenizer=tokenizer,
    repo_name=REPO_NAME,
    private=False,
    training_params=training_params,
)

# 2. Push Model Card README
api.upload_file(
    path_or_fileobj=model_card.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=REPO_NAME,
)
print(f"Model Card hochgeladen.")
print(f"\nModell veroeffentlicht: {url}")

In [ ]:
# === SUMMARY ===
print("=" * 70)
print(f"  FINAL MODEL TRAINING COMPLETE")
print("=" * 70)
print(f"  Model:           {REPO_NAME}")
print(f"  Base:            {MODEL_ID}")
print(f"  HPT Source:      Trial {study.best_trial.number} (CV F1={study.best_value:.4f})")
print(f"  Train:           {len(train_df)} Artikel")
print(f"  Validation:      {len(val_df)} Artikel")
print(f"  Test:            {len(test_df)} Artikel")
print(f"  Epochs:          {NUM_EPOCHS} (best: {trainer.state.best_model_checkpoint})")
print(f"  F1 Macro:        {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:     {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  Precision Macro: {metrics['precision_macro']:.4f}")
print(f"  Recall Macro:    {metrics['recall_macro']:.4f}")
print(f"  Dauer:           {timer.duration_formatted}")
print(f"  Report:          {report_path}")
print(f"  HuggingFace:     {url}")
print("=" * 70)

In [ ]:
# === CLEANUP ===
import shutil

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    print("Checkpoint-Dateien geloescht.")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU VRAM frei: {free_mem:.1f} GB")

print(f"\nErgebnisse auf Google Drive:")
print(f"  Report: {report_path}")
print(f"\nModell auf HuggingFace:")
print(f"  {url}")
print(f"\nFertig. Runtime kann beendet werden.")